# Spam Guard Training

In [2]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from tqdm import tqdm


In [3]:
print("Loading vectors...")
x_tr = joblib.load('x_training_vector.pkl')
x_te = joblib.load('x_testing_vector.pkl')
y_tr = joblib.load('y_training_vector.pkl')
y_te = joblib.load('y_testing_vector.pkl')

print(f'x_tr shape: {x_tr.shape}')
print(f'x_te shape: {x_te.shape}')
print(f'y_tr length: {len(y_tr)}')
print(f'y_te length: {len(y_te)}')

Loading vectors...
x_tr shape: (326222, 5000)
x_te shape: (102111, 5000)
y_tr length: 326222
y_te length: 102111


## Feature Selection
Use chi-square to keep top 1000 features.

In [4]:
print("Running feature selection...")
sel = SelectKBest(chi2, k=1000)
x_tr_s = sel.fit_transform(x_tr, y_tr)
x_te_s = sel.transform(x_te)

print(f'x_tr_s shape: {x_tr_s.shape}')
print(f'x_te_s shape: {x_te_s.shape}')

Running feature selection...
x_tr_s shape: (326222, 1000)
x_te_s shape: (102111, 1000)


In [5]:
# Train models on a small sample
x_sample = x_tr_s[:5000]
y_sample = y_tr[:5000]

print(f'x_sample shape: {x_sample.shape}')
print(f'y_sample length: {len(y_sample)}')

x_sample shape: (5000, 1000)
y_sample length: 5000


## Grid Search Definitions

In [9]:
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs'],
    },
    cv = 5,
    scoring = 'f1',
    n_jobs = -1,
)

print('Defined Logistic Regression grid.')

Defined Logistic Regression grid.


In [16]:
svm_poly_grid = GridSearchCV(
    SVC(kernel='poly', probability=True, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'degree': [2, 3],
        'gamma': ['scale', 'auto'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined SVM Poly grid.')

Defined SVM Poly grid.


In [17]:
svm_rbf_grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined SVM RBF grid.')

Defined SVM RBF grid.


In [18]:
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['minkowski', 'manhattan'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined KNN grid.')

Defined KNN grid.


In [19]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        'n_estimators': [100, 200],
        'max_depth': [None, 20],
        'min_samples_split': [2, 5],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined Random Forest grid.')

Defined Random Forest grid.


In [28]:
ridge_calibrated = CalibratedClassifierCV(
    RidgeClassifier(class_weight='balanced', random_state=42),
    method='sigmoid',
    cv=5,
)

ridge_grid = GridSearchCV(
    ridge_calibrated,
    {
        'estimator__alpha': [0.1, 1.0, 10.0],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined Ridge Classifier grid.')

Defined Ridge Classifier grid.


In [29]:
grid_jobs = {
    "Logistic_Regression": lr_grid,
    "SVM_Poly": svm_poly_grid,
    "SVM RBF": svm_rbf_grid,
    "KNN": knn_grid,
    "Random_Forest": rf_grid,
    "Ridge": ridge_grid
}

print('Tuning models...')
for model_name, grid in tqdm(grid_jobs.items(), total=len(grid_jobs), desc='Grid search'):
    grid.fit(x_sample, y_sample)
    print(f'  Tuned: {model_name}')
print('All grid searches complete.')

Tuning models...


Grid search:  17%|███████████▊                                                           | 1/6 [00:05<00:29,  5.94s/it]

  Tuned: Logistic_Regression


Grid search:  33%|███████████████████████▎                                              | 2/6 [05:16<12:20, 185.05s/it]

  Tuned: SVM_Poly


...

  Tuned: KNN


Grid search:  83%|███████████████████████████████████████████████████████████▏           | 5/6 [08:38<01:20, 80.75s/it]

  Tuned: Random_Forest


Grid search: 100%|███████████████████████████████████████████████████████████████████████| 6/6 [08:39<00:00, 86.65s/it]

  Tuned: Ridge
All grid searches complete.


In [30]:
print('LR best params:', lr_grid.best_params_)
print('SVM Poly best params:', svm_poly_grid.best_params_)
print('SVM RBF best params:', svm_rbf_grid.best_params_)
print('KNN best params:', knn_grid.best_params_)
print('RF best params:', rf_grid.best_params_)
print('Ridge best params:', ridge_grid.best_params_)

LR best params: {'C': 10, 'solver': 'liblinear'}
SVM Poly best params: {'C': 10, 'degree': 2, 'gamma': 'scale'}
SVM RBF best params: {'C': 1, 'gamma': 'scale'}
KNN best params: {'metric': 'minkowski', 'n_neighbors': 3, 'weights': 'distance'}
RF best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Ridge best params: {'estimator__alpha': 1.0}


In [31]:
best_models = {
    'Logistic Regression': lr_grid.best_estimator_,
    'SVM Poly': svm_poly_grid.best_estimator_,
    'SVM RBF': svm_rbf_grid.best_estimator_,
    'KNN': knn_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_,
    'Ridge Classifier': ridge_grid.best_estimator_,
}

print('Best model objects assembled.')

Best model objects assembled.


In [ ]:
print('Training tuned models on full training set...')
for model_name, model in tqdm(best_models.items(), total=len(best_models), desc='Training base models'):
    model.fit(x_tr_s, y_tr)
    print(f'  Trained: {model_name}')
    joblib.dump(model, f'model_{model_name}.pkl')

Training tuned models on full training set...
  Trained: Logistic Regression


In [27]:
print('Building ensemble...')
ensemble = VotingClassifier(
    estimators=[
        ('lr', best_models['Logistic Regression']),
        ('svm_poly', best_models['SVM Poly']),
        ('svm_rbf', best_models['SVM RBF']),
        ('knn', best_models['KNN']),
        ('rf', best_models['Random Forest']),
        ('ridge', best_models['Ridge Classifier']),
    ],
    voting='soft',
)
ensemble.fit(x_tr_s, y_tr)
print('Ensemble training complete.')

Building ensemble...


KeyboardInterrupt: 

In [ ]:
joblib.dump(ensemble, '/kaggle/working/model.pkl')
joblib.dump(sel, '/kaggle/working/sel.pkl')
print(f"Success! Model and Selector saved")